### Импорты и загрузка данных

In [1]:
import sys
sys.path.append('../src')

import torch
import pandas as pd
import numpy as np
from pathlib import Path

from dataset import build_item_vocab
from reward_model import RewardModel, create_reward_model, predict_reward

PROC = Path('../data/processed')
RESULTS = Path('../results')

train_df = pd.read_parquet(PROC / 'train.parquet')
item2idx, idx2item = build_item_vocab(train_df)
vocab_size = len(item2idx) + 2
print(f"Vocab size: {vocab_size}")

Vocab size: 14937


### Создание RM без инициализации

In [2]:
model_random = create_reward_model(vocab_size=vocab_size)
print(f"\nМодель создана (random init)")
print(f"Тип: {type(model_random).__name__}")

# Рассчёт параметров
n_params = sum(p.numel() for p in model_random.parameters())
print(f"Параметров всего: {n_params:,}")

# Разбиение: backbone vs head
n_backbone = sum(p.numel() for p in model_random.backbone.parameters())
n_head = sum(p.numel() for p in model_random.head.parameters())
print(f"  Backbone: {n_backbone:,}")
print(f"  Head:     {n_head:,}")


Модель создана (random init)
Тип: RewardModel
Параметров всего: 2,331,777
  Backbone: 2,315,136
  Head:     16,641


### Рождение RM с инииализацией из GPT-2 v2

In [3]:
model_pretrained = create_reward_model(
    vocab_size=vocab_size,
    init_from=RESULTS / 'checkpoints_v2' / 'best_model.pt',
)
print(f"\nМодель создана (pretrained backbone)")

# Проверка: backbone весов совпадает с GPT-2 v2?
gpt2_state = torch.load(RESULTS / 'checkpoints_v2' / 'best_model.pt', map_location='cpu')
gpt2_wte = gpt2_state['transformer.wte.weight']
rm_wte = model_pretrained.backbone.wte.weight
print(f"\nGPT-2 wte.shape: {gpt2_wte.shape}")
print(f"RM    wte.shape: {rm_wte.shape}")
print(f"Совпадают: {torch.allclose(gpt2_wte, rm_wte)}")

# Проверка: голова случайная (не совпадает с GPT-2 lm_head)
gpt2_lmhead = gpt2_state['lm_head.weight']
print(f"\nGPT-2 lm_head.shape: {gpt2_lmhead.shape}")   # [vocab_size, n_embd]
print(f"RM head — это Sequential, а не Linear — видим:")
for name, p in model_pretrained.head.named_parameters():
    print(f"  {name}: {tuple(p.shape)}")

[create_reward_model] Загружено параметров backbone: 28

Модель создана (pretrained backbone)

GPT-2 wte.shape: torch.Size([14937, 128])
RM    wte.shape: torch.Size([14937, 128])
Совпадают: True

GPT-2 lm_head.shape: torch.Size([14937, 128])
RM head — это Sequential, а не Linear — видим:
  0.weight: (128, 128)
  0.bias: (128,)
  3.weight: (1, 128)
  3.bias: (1,)


### Forward pass (вход случайный)

In [4]:
# Случайный батч из 4 примеров
B, L = 4, 50
input_ids = torch.randint(2, vocab_size, (B, L))   # от 2 (не PAD/UNK) до vocab_size

# Forward через обе модели
model_random.eval()
model_pretrained.eval()
with torch.no_grad():
    logits_random = model_random(input_ids)
    logits_pretrained = model_pretrained(input_ids)

print(f"input_ids.shape: {input_ids.shape}")
print(f"\nlogits (random init).shape: {logits_random.shape}")
print(f"logits (pretrained).shape:  {logits_pretrained.shape}")
print(f"\nlogits (random init):    {logits_random.tolist()}")
print(f"logits (pretrained):     {logits_pretrained.tolist()}")
print(f"\nsigmoid(random):  {torch.sigmoid(logits_random).tolist()}")
print(f"sigmoid(pretrained): {torch.sigmoid(logits_pretrained).tolist()}")

# Проверки
assert logits_random.shape == (B,), "logits должен быть (B,)"
assert logits_pretrained.shape == (B,)
assert (torch.sigmoid(logits_random) >= 0).all() and (torch.sigmoid(logits_random) <= 1).all()
print("\nВсе проверки пройдены")

input_ids.shape: torch.Size([4, 50])

logits (random init).shape: torch.Size([4])
logits (pretrained).shape:  torch.Size([4])

logits (random init):    [-0.332898885011673, -0.21149367094039917, -0.1126309409737587, -0.22994518280029297]
logits (pretrained):     [-0.015492264181375504, -0.07232672721147537, -0.12563325464725494, 0.047269366681575775]

sigmoid(random):  [0.4175354540348053, 0.4473227560520172, 0.4718720018863678, 0.44276565313339233]
sigmoid(pretrained): [0.49612700939178467, 0.48192620277404785, 0.46863290667533875, 0.5118151307106018]

Все проверки пройдены


### Реальный пример из RM-индекса

In [6]:
from rm_index import RMIndex, RMDataset

# Загрузка объединённого датафрейма
val_df = pd.read_parquet(PROC / 'val.parquet')
test_df = pd.read_parquet(PROC / 'test.parquet')
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

# Построение индекса
idx = RMIndex(all_df, item2idx, t1='2021-07-24', t2='2021-08-04')

# Датасет
train_ds = RMDataset(idx, idx.train_mask, max_len=50)

# 4 примера из train
batch = [train_ds[i] for i in range(4)]
input_ids_batch = torch.stack([b[0] for b in batch])   # (4, 50)
rewards_true = torch.stack([b[1] for b in batch])      # (4,)
print(f"input_ids_batch.shape: {input_ids_batch.shape}")
print(f"rewards_true: {rewards_true.tolist()}")

# Прогон через RM
with torch.no_grad():
    logits = model_pretrained(input_ids_batch)
    rewards_pred = torch.sigmoid(logits)

print(f"\nrewards_true: {rewards_true.tolist()}")
print(f"rewards_pred: {rewards_pred.tolist()}")

RMIndex: 4,632,197 событий
  train: 2,789,962
  val:   371,021
  test:  474,130
RMDataset: 2,789,962 примеров
input_ids_batch.shape: torch.Size([4, 50])
rewards_true: [0.029999999329447746, 0.019999999552965164, 0.0, 0.0]

rewards_true: [0.029999999329447746, 0.019999999552965164, 0.0, 0.0]
rewards_pred: [0.5969213247299194, 0.5469598770141602, 0.5949365496635437, 0.6413766145706177]


*Предсказания пока случайные, т. к. модель на данный момент не обучена*